# Is the emergency model really that good? Two checks

The emergency model scored mAP@0.5 = 0.995 on the dataset's validation split. That is suspiciously high, so this notebook checks it:

1. **Near-duplicates:** how many validation images are almost the same picture as a training image? (If many are, the validation score mostly measures memory, not generalisation.)
2. **A real video the model never saw:** the Pexels highway video, which contains a police car with blue lights, run through the app's full pipeline with the emergency model.

**Inputs:** the output of the training notebook (for `emergency_yolov8s.pt`) and the EV vs NEV dataset. Accelerator: GPU T4. Internet: On.

In [ ]:
!pip install -q ultralytics
import glob, json, os, sys
import cv2
import numpy as np
%cd /tmp
!rm -rf smart-traffic-management && git clone -q https://github.com/pavanadityav-ai/smart-traffic-management.git
%cd /tmp/smart-traffic-management
sys.path.insert(0, ".")
WEIGHTS = glob.glob("/kaggle/input/**/emergency_yolov8s.pt", recursive=True)[0]
BASE = os.path.dirname(glob.glob("/kaggle/input/**/data.yaml", recursive=True)[0])
OUT = "/kaggle/working/check"
os.makedirs(OUT, exist_ok=True)
print(WEIGHTS, BASE)

## 1. Near-duplicate check (difference hash)

In [ ]:
def dhash(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA)
    bits = (small[:, 1:] > small[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

train = sorted(glob.glob(f"{BASE}/train/images/*"))
val = sorted(glob.glob(f"{BASE}/val/images/*"))
th = np.array([dhash(p) for p in train], dtype=np.uint64)
vh = np.array([dhash(p) for p in val], dtype=np.uint64)

def popcount(x):
    x = x.copy(); c = np.zeros(x.shape, dtype=np.uint8)
    for _ in range(64):
        c += (x & np.uint64(1)).astype(np.uint8); x >>= np.uint64(1)
    return c

nearest = np.array([popcount(th ^ h).min() for h in vh])
dup = {f"distance <= {d} bits": round(float((nearest <= d).mean() * 100), 1) for d in (0, 3, 6, 10)}
print("validation images:", len(val), "| % with a near-copy in training:", dup)

## 2. Real video the model never saw

In [ ]:
from traffic.detection import CombinedDetector, YoloDetector
from traffic.pipeline import TrafficPipeline

!wget -q -U "Mozilla/5.0" https://videos.pexels.com/video-files/2103099/2103099-uhd_2560_1440_30fps.mp4 -O /tmp/highway.mp4
cap = cv2.VideoCapture("/tmp/highway.mp4")
fps = cap.get(cv2.CAP_PROP_FPS) or 25
em_model = YoloDetector(WEIGHTS, 0.4)
print("emergency model classes:", em_model.class_names)
pipe = TrafficPipeline(CombinedDetector(YoloDetector("yolov8s.pt", 0.4), em_model), n_lanes=4)
em_frames, saved, i = [], 0, 0
while i < 900:
    ok, frame = cap.read()
    if not ok:
        break
    i += 1
    frame = cv2.resize(frame, (1280, 720))
    annotated, state = pipe.process(frame, i / fps)
    raw, _ = em_model.detect(frame)
    if any(d.emergency for d in raw):
        em_frames.append(i)
        if saved < 3 and i % 60 == 0:
            cv2.imwrite(f"{OUT}/emergency_frame_{i:04d}.jpg", annotated); saved += 1
    if i in (450, 675):
        cv2.imwrite(f"{OUT}/frame_{i:04d}.jpg", annotated)

seconds = sorted({round(f / fps) for f in em_frames})
summary = {"near_duplicates_val_in_train_percent": dup,
           "video_frames": i, "frames_with_emergency_detection": len(em_frames),
           "seconds_with_emergency_detection": seconds,
           "alerts": pipe.alerts.alerts}
json.dump(summary, open(f"{OUT}/check_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
from IPython.display import Image, display
for f in sorted(os.listdir(OUT)):
    if f.endswith(".jpg"):
        print(f); display(Image(f"{OUT}/{f}", width=900))